# 04 - Machine Learning Models: El Clasico Prediction

**Dataset:** 202 La Liga El Clasico matches (1929-2026), 68 features

---

## Models Built

| # | Model | Task | Target |
|---|-------|------|--------|
| 1 | Logistic Regression | Classification | Match result (H/D/A) |
| 2 | Random Forest Classifier | Classification | Match result (H/D/A) |
| 3 | K-Nearest Neighbors | Classification | RM Win / Loss / Draw |
| 4 | Support Vector Machine | Classification | RM Win / Loss / Draw |
| 5 | Decision Tree | Classification | Goal Category |
| 6 | Linear Regression | Regression | Total goals |
| 7 | Random Forest Regressor | Regression | Total goals |
| 8 | XGBoost | Classification | Match result (H/D/A) |
| 9 | Ensemble Voting Classifier | Classification | Match result (H/D/A) |
| 10 | GridSearchCV Hyperparameter Tuning | Optimization | Best Random Forest |
| 11 | Model Comparison | All models | Accuracy / MAE comparison |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score
)

# Classification models
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)

df = pd.read_csv('../data/final/el_clasico_final.csv')
print(f'Loaded: {df.shape[0]} matches, {df.shape[1]} features')

---
## Data Preparation

In [ ]:
# Select features that would be available BEFORE the match
# (We cannot use goals, shots etc. as those happen during the match)

# Build historical features per row:
# rolling averages of past matches for each team

ml_df = df.dropna(subset=['year', 'result']).sort_values('year').reset_index(drop=True).copy()

# Compute rolling averages over the last 5 matches
window = 5
for col in ['rm_goals', 'barca_goals', 'total_goals', 'home_goals', 'away_goals']:
    ml_df[f'{col}_roll{window}'] = ml_df[col].rolling(window=window, min_periods=1).mean().shift(1)

# Rolling win rate for RM in last N matches
ml_df['rm_win_flag'] = (ml_df['rm_result'] == 'RM Win').astype(int)
ml_df['barca_win_flag'] = (ml_df['rm_result'] == 'RM Loss').astype(int)
ml_df['draw_flag'] = (ml_df['rm_result'] == 'Draw').astype(int)

ml_df['rm_winrate_roll5'] = ml_df['rm_win_flag'].rolling(window=window, min_periods=1).mean().shift(1)
ml_df['barca_winrate_roll5'] = ml_df['barca_win_flag'].rolling(window=window, min_periods=1).mean().shift(1)
ml_df['draw_rate_roll5'] = ml_df['draw_flag'].rolling(window=window, min_periods=1).mean().shift(1)

# Cumulative stats up to that point
ml_df['rm_cum_wins'] = ml_df['rm_win_flag'].cumsum().shift(1).fillna(0)
ml_df['barca_cum_wins'] = ml_df['barca_win_flag'].cumsum().shift(1).fillna(0)
ml_df['match_index'] = range(len(ml_df))
ml_df['rm_cum_winrate'] = (ml_df['rm_cum_wins'] / (ml_df['match_index'] + 1)).fillna(0)

# Encode venue
ml_df['is_rm_home'] = (ml_df['home_team'] == 'Real Madrid').astype(int)

# Year and decade as numeric
ml_df['decade_num'] = (ml_df['year'] // 10) * 10

print(f'Prepared: {ml_df.shape[0]} rows')
print(f'New features: rolling averages, win rates, cumulative stats')

In [ ]:
# Define feature sets

FEATURES = [
    'year', 'is_rm_home',
    'rm_goals_roll5', 'barca_goals_roll5', 'total_goals_roll5',
    'home_goals_roll5', 'away_goals_roll5',
    'rm_winrate_roll5', 'barca_winrate_roll5', 'draw_rate_roll5',
    'rm_cum_winrate',
]

# Drop rows with NaN in features (first few rows won't have rolling stats)
ml_ready = ml_df.dropna(subset=FEATURES).copy()
print(f'Rows ready for ML: {len(ml_ready)}')
print(f'Features: {FEATURES}')
print()

# Target distributions
print('Target: result (H/D/A)')
print(ml_ready['result'].value_counts().to_string())
print()
print('Target: rm_result')
print(ml_ready['rm_result'].value_counts().to_string())

In [ ]:
# Prepare X and y
X = ml_ready[FEATURES].values
y_result = ml_ready['result'].values          # H, D, A
y_rm = ml_ready['rm_result'].values            # RM Win, RM Loss, Draw
y_goals = ml_ready['total_goals'].values       # regression target
y_cat = ml_ready['goal_category'].values       # classification: Goalless/Low/Medium/High

# Encode string targets
le_result = LabelEncoder()
y_result_enc = le_result.fit_transform(y_result)

le_rm = LabelEncoder()
y_rm_enc = le_rm.fit_transform(y_rm)

le_cat = LabelEncoder()
y_cat_enc = le_cat.fit_transform(y_cat)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split (80/20)
# Use last 20% as test set (time-based split, more realistic)
split_idx = int(len(X_scaled) * 0.8)

X_train, X_test = X_scaled[:split_idx], X_scaled[split_idx:]
y_train_result, y_test_result = y_result_enc[:split_idx], y_result_enc[split_idx:]
y_train_rm, y_test_rm = y_rm_enc[:split_idx], y_rm_enc[split_idx:]
y_train_goals, y_test_goals = y_goals[:split_idx], y_goals[split_idx:]
y_train_cat, y_test_cat = y_cat_enc[:split_idx], y_cat_enc[split_idx:]

print(f'Train size: {len(X_train)}')
print(f'Test size:  {len(X_test)}')

In [ ]:
# Helper function for confusion matrix plotting
def plot_confusion_matrix(y_true, y_pred, labels, title, ax):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(title, fontsize=13, fontweight='bold')

# Store results for comparison
results_table = []

---
## Model 1: Logistic Regression (Match Result H/D/A)

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train_result)

y_pred_lr = lr.predict(X_test)
acc_lr = accuracy_score(y_test_result, y_pred_lr)

# Cross-validation
cv_scores_lr = cross_val_score(lr, X_scaled, y_result_enc, cv=5, scoring='accuracy')

print('LOGISTIC REGRESSION: Match Result (H/D/A)')
print('=' * 50)
print(f'Test Accuracy:  {acc_lr:.4f}')
print(f'CV Accuracy:    {cv_scores_lr.mean():.4f} (+/- {cv_scores_lr.std():.4f})')
print()
print('Classification Report:')
print(classification_report(y_test_result, y_pred_lr, target_names=le_result.classes_))

results_table.append({'Model': 'Logistic Regression', 'Task': 'Result (H/D/A)',
                       'Test Accuracy': acc_lr, 'CV Accuracy': cv_scores_lr.mean()})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_confusion_matrix(y_test_result, y_pred_lr, le_result.classes_,
                      'Logistic Regression: Confusion Matrix', axes[0])

# Feature importance (coefficients)
coef_df = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': np.abs(lr.coef_).mean(axis=0)
}).sort_values('Importance', ascending=True)

axes[1].barh(coef_df['Feature'], coef_df['Importance'], color='#42A5F5', edgecolor='black')
axes[1].set_title('Feature Importance (Avg |Coefficient|)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

---
## Model 2: Random Forest Classifier (Match Result H/D/A)

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
rf.fit(X_train, y_train_result)

y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test_result, y_pred_rf)

cv_scores_rf = cross_val_score(rf, X_scaled, y_result_enc, cv=5, scoring='accuracy')

print('RANDOM FOREST CLASSIFIER: Match Result (H/D/A)')
print('=' * 50)
print(f'Test Accuracy:  {acc_rf:.4f}')
print(f'CV Accuracy:    {cv_scores_rf.mean():.4f} (+/- {cv_scores_rf.std():.4f})')
print()
print('Classification Report:')
print(classification_report(y_test_result, y_pred_rf, target_names=le_result.classes_))

results_table.append({'Model': 'Random Forest', 'Task': 'Result (H/D/A)',
                       'Test Accuracy': acc_rf, 'CV Accuracy': cv_scores_rf.mean()})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_confusion_matrix(y_test_result, y_pred_rf, le_result.classes_,
                      'Random Forest: Confusion Matrix', axes[0])

# Feature importance
fi_df = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=True)

axes[1].barh(fi_df['Feature'], fi_df['Importance'], color='#4CAF50', edgecolor='black')
axes[1].set_title('Random Forest: Feature Importance', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

---
## Model 3: K-Nearest Neighbors (RM Win/Loss/Draw)

In [ ]:
# Find best K using cross-validation
k_range = range(3, 21, 2)
k_scores = []

for k in k_range:
    knn_temp = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn_temp, X_scaled, y_rm_enc, cv=5, scoring='accuracy')
    k_scores.append(scores.mean())

best_k = list(k_range)[np.argmax(k_scores)]
print(f'Best K = {best_k} (CV Accuracy = {max(k_scores):.4f})')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(k_range), k_scores, 'o-', color='#9C27B0', linewidth=2, markersize=8)
ax.axvline(best_k, color='red', linestyle='--', alpha=0.5, label=f'Best K = {best_k}')
ax.set_xlabel('K (Number of Neighbors)')
ax.set_ylabel('CV Accuracy')
ax.set_title('KNN: Finding Optimal K', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train, y_train_rm)

y_pred_knn = knn.predict(X_test)
acc_knn = accuracy_score(y_test_rm, y_pred_knn)

cv_scores_knn = cross_val_score(knn, X_scaled, y_rm_enc, cv=5, scoring='accuracy')

print(f'KNN (K={best_k}): RM Win/Loss/Draw')
print('=' * 50)
print(f'Test Accuracy:  {acc_knn:.4f}')
print(f'CV Accuracy:    {cv_scores_knn.mean():.4f} (+/- {cv_scores_knn.std():.4f})')
print()
print('Classification Report:')
print(classification_report(y_test_rm, y_pred_knn, target_names=le_rm.classes_))

results_table.append({'Model': f'KNN (K={best_k})', 'Task': 'RM Result',
                       'Test Accuracy': acc_knn, 'CV Accuracy': cv_scores_knn.mean()})

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
plot_confusion_matrix(y_test_rm, y_pred_knn, le_rm.classes_,
                      f'KNN (K={best_k}): Confusion Matrix', ax)
plt.tight_layout()
plt.show()

---
## Model 4: Support Vector Machine (RM Win/Loss/Draw)

In [ ]:
svm = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm.fit(X_train, y_train_rm)

y_pred_svm = svm.predict(X_test)
acc_svm = accuracy_score(y_test_rm, y_pred_svm)

cv_scores_svm = cross_val_score(svm, X_scaled, y_rm_enc, cv=5, scoring='accuracy')

print('SVM (RBF Kernel): RM Win/Loss/Draw')
print('=' * 50)
print(f'Test Accuracy:  {acc_svm:.4f}')
print(f'CV Accuracy:    {cv_scores_svm.mean():.4f} (+/- {cv_scores_svm.std():.4f})')
print()
print('Classification Report:')
print(classification_report(y_test_rm, y_pred_svm, target_names=le_rm.classes_))

results_table.append({'Model': 'SVM (RBF)', 'Task': 'RM Result',
                       'Test Accuracy': acc_svm, 'CV Accuracy': cv_scores_svm.mean()})

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
plot_confusion_matrix(y_test_rm, y_pred_svm, le_rm.classes_,
                      'SVM: Confusion Matrix', ax)
plt.tight_layout()
plt.show()

---
## Model 5: Decision Tree (Goal Category)

In [ ]:
dt = DecisionTreeClassifier(max_depth=4, random_state=42, min_samples_leaf=5)
dt.fit(X_train, y_train_cat)

y_pred_dt = dt.predict(X_test)
acc_dt = accuracy_score(y_test_cat, y_pred_dt)

cv_scores_dt = cross_val_score(dt, X_scaled, y_cat_enc, cv=5, scoring='accuracy')

print('DECISION TREE: Goal Category (Goalless/Low/Medium/High)')
print('=' * 50)
print(f'Test Accuracy:  {acc_dt:.4f}')
print(f'CV Accuracy:    {cv_scores_dt.mean():.4f} (+/- {cv_scores_dt.std():.4f})')
print()
print('Classification Report:')
print(classification_report(y_test_cat, y_pred_dt, target_names=le_cat.classes_))

results_table.append({'Model': 'Decision Tree', 'Task': 'Goal Category',
                       'Test Accuracy': acc_dt, 'CV Accuracy': cv_scores_dt.mean()})

In [ ]:
# Print the decision tree rules
print('Decision Tree Rules:')
print('=' * 50)
tree_rules = export_text(dt, feature_names=FEATURES, max_depth=4)
print(tree_rules)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_confusion_matrix(y_test_cat, y_pred_dt, le_cat.classes_,
                      'Decision Tree: Confusion Matrix', axes[0])

# Feature importance
dt_fi = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': dt.feature_importances_
}).sort_values('Importance', ascending=True)

axes[1].barh(dt_fi['Feature'], dt_fi['Importance'], color='#FF9800', edgecolor='black')
axes[1].set_title('Decision Tree: Feature Importance', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

---
## Model 6: Linear Regression (Total Goals)

In [ ]:
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train_goals)

y_pred_lin = lin_reg.predict(X_test)

mae_lin = mean_absolute_error(y_test_goals, y_pred_lin)
rmse_lin = np.sqrt(mean_squared_error(y_test_goals, y_pred_lin))
r2_lin = r2_score(y_test_goals, y_pred_lin)

print('LINEAR REGRESSION: Total Goals Prediction')
print('=' * 50)
print(f'MAE:   {mae_lin:.3f}')
print(f'RMSE:  {rmse_lin:.3f}')
print(f'R2:    {r2_lin:.3f}')
print()

# Coefficients
coef_lr_df = pd.DataFrame({
    'Feature': FEATURES,
    'Coefficient': lin_reg.coef_
}).sort_values('Coefficient', key=abs, ascending=False)
print('Feature Coefficients:')
print(coef_lr_df.to_string(index=False))

results_table.append({'Model': 'Linear Regression', 'Task': 'Total Goals (Reg)',
                       'Test Accuracy': r2_lin, 'CV Accuracy': None})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Actual vs Predicted
axes[0].scatter(y_test_goals, y_pred_lin, c='#2196F3', s=60, alpha=0.7, edgecolors='black')
axes[0].plot([0, 10], [0, 10], 'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Total Goals')
axes[0].set_ylabel('Predicted Total Goals')
axes[0].set_title(f'Linear Regression: Actual vs Predicted (R2={r2_lin:.3f})', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)

# Residuals
residuals = y_test_goals - y_pred_lin
axes[1].scatter(y_pred_lin, residuals, c='#FF5722', s=60, alpha=0.7, edgecolors='black')
axes[1].axhline(0, color='black', linestyle='--')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Residual')
axes[1].set_title(f'Residual Plot (MAE={mae_lin:.3f})', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

---
## Model 7: Random Forest Regressor (Total Goals)

In [ ]:
rf_reg = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
rf_reg.fit(X_train, y_train_goals)

y_pred_rfr = rf_reg.predict(X_test)

mae_rfr = mean_absolute_error(y_test_goals, y_pred_rfr)
rmse_rfr = np.sqrt(mean_squared_error(y_test_goals, y_pred_rfr))
r2_rfr = r2_score(y_test_goals, y_pred_rfr)

print('RANDOM FOREST REGRESSOR: Total Goals Prediction')
print('=' * 50)
print(f'MAE:   {mae_rfr:.3f}')
print(f'RMSE:  {rmse_rfr:.3f}')
print(f'R2:    {r2_rfr:.3f}')

results_table.append({'Model': 'RF Regressor', 'Task': 'Total Goals (Reg)',
                       'Test Accuracy': r2_rfr, 'CV Accuracy': None})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Actual vs Predicted
axes[0].scatter(y_test_goals, y_pred_rfr, c='#4CAF50', s=60, alpha=0.7, edgecolors='black')
axes[0].plot([0, 10], [0, 10], 'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Total Goals')
axes[0].set_ylabel('Predicted Total Goals')
axes[0].set_title(f'RF Regressor: Actual vs Predicted (R2={r2_rfr:.3f})', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)

# Feature importance
rfr_fi = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': rf_reg.feature_importances_
}).sort_values('Importance', ascending=True)

axes[1].barh(rfr_fi['Feature'], rfr_fi['Importance'], color='#4CAF50', edgecolor='black')
axes[1].set_title('RF Regressor: Feature Importance', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

---
## Model 8: XGBoost Classifier

In [ ]:
xgb = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    use_label_encoder=False, eval_metric='mlogloss', random_state=42
)
xgb.fit(X_train, y_train_result)

y_pred_xgb = xgb.predict(X_test)
acc_xgb = accuracy_score(y_test_result, y_pred_xgb)
cv_scores_xgb = cross_val_score(xgb, X_scaled, y_result_enc, cv=5, scoring='accuracy')

print('XGBOOST CLASSIFIER')
print('=' * 50)
print(f'Test Accuracy:  {acc_xgb:.4f}')
print(f'CV Accuracy:    {cv_scores_xgb.mean():.4f} (+/- {cv_scores_xgb.std():.4f})')
print()
print('Classification Report:')
print(classification_report(y_test_result, y_pred_xgb, target_names=le_result.classes_))

results_table.append({'Model': 'XGBoost', 'Task': 'Result (H/D/A)',
                       'Test Accuracy': acc_xgb, 'CV Accuracy': cv_scores_xgb.mean()})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_confusion_matrix(y_test_result, y_pred_xgb, le_result.classes_,
                      'XGBoost: Confusion Matrix', axes[0])

xgb_fi = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': xgb.feature_importances_
}).sort_values('Importance', ascending=True)

axes[1].barh(xgb_fi['Feature'], xgb_fi['Importance'], color='#E91E63', edgecolor='black')
axes[1].set_title('XGBoost: Feature Importance', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

---
## Model 9: Ensemble Voting Classifier

In [ ]:
# Combine the best classifiers into a voting ensemble
voting_clf = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=1000, random_state=42)),
        ('rf', RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)),
        ('xgb', XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                              use_label_encoder=False, eval_metric='mlogloss', random_state=42)),
        ('knn', KNeighborsClassifier(n_neighbors=best_k)),
    ],
    voting='hard'
)

voting_clf.fit(X_train, y_train_result)
y_pred_vote = voting_clf.predict(X_test)
acc_vote = accuracy_score(y_test_result, y_pred_vote)
cv_scores_vote = cross_val_score(voting_clf, X_scaled, y_result_enc, cv=5, scoring='accuracy')

print('ENSEMBLE VOTING CLASSIFIER')
print('=' * 50)
print(f'Test Accuracy:  {acc_vote:.4f}')
print(f'CV Accuracy:    {cv_scores_vote.mean():.4f} (+/- {cv_scores_vote.std():.4f})')
print()
print('Classification Report:')
print(classification_report(y_test_result, y_pred_vote, target_names=le_result.classes_))

results_table.append({'Model': 'Ensemble Voting', 'Task': 'Result (H/D/A)',
                       'Test Accuracy': acc_vote, 'CV Accuracy': cv_scores_vote.mean()})

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
plot_confusion_matrix(y_test_result, y_pred_vote, le_result.classes_,
                      'Ensemble Voting: Confusion Matrix', ax)
plt.tight_layout()
plt.show()

---
## Model 10: GridSearchCV Hyperparameter Tuning

In [ ]:
# Hyperparameter tuning for Random Forest
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}

print('Running GridSearchCV on Random Forest...')
print(f'Parameter grid: {len(param_grid["n_estimators"])} x {len(param_grid["max_depth"])} x {len(param_grid["min_samples_split"])} x {len(param_grid["min_samples_leaf"])} = {3*4*3*3} combinations')
print('This may take a minute...\n')

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)

grid_search.fit(X_scaled, y_result_enc)

print('GRIDSEARCHCV RESULTS')
print('=' * 50)
print(f'Best Parameters: {grid_search.best_params_}')
print(f'Best CV Accuracy: {grid_search.best_score_:.4f}')
print()

# Evaluate best model on test set
best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)
acc_best = accuracy_score(y_test_result, y_pred_best)
print(f'Test Accuracy (Tuned RF): {acc_best:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test_result, y_pred_best, target_names=le_result.classes_))

results_table.append({'Model': 'Tuned RF (GridSearch)', 'Task': 'Result (H/D/A)',
                       'Test Accuracy': acc_best, 'CV Accuracy': grid_search.best_score_})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_confusion_matrix(y_test_result, y_pred_best, le_result.classes_,
                      'Tuned RF: Confusion Matrix', axes[0])

# Show top GridSearch results
cv_results = pd.DataFrame(grid_search.cv_results_)
top10_params = cv_results.nsmallest(10, 'rank_test_score')[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']]
top10_params.columns = ['Parameters', 'Mean CV Score', 'Std', 'Rank']

axes[1].barh(range(len(top10_params)), top10_params['Mean CV Score'],
             xerr=top10_params['Std'], color='#4CAF50', edgecolor='black', alpha=0.8)
axes[1].set_yticks(range(len(top10_params)))
axes[1].set_yticklabels([f'Rank {r}' for r in top10_params['Rank']], fontsize=9)
axes[1].set_xlabel('CV Accuracy')
axes[1].set_title('Top 10 GridSearch Configurations', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print('\nTop 5 parameter sets:')
print(top10_params.head().to_string(index=False))

---
## Model 11: Final Comparison (All Models)

In [ ]:
results_df = pd.DataFrame(results_table)
print('MODEL COMPARISON')
print('=' * 70)
print(results_df.to_string(index=False))

In [ ]:
# Classification models comparison chart
clf_results = results_df[results_df['Task'] != 'Total Goals (Reg)'].copy()

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(clf_results))
w = 0.35

bars1 = ax.bar(x - w/2, clf_results['Test Accuracy'], w,
               label='Test Accuracy', color='#2196F3', edgecolor='black')
bars2 = ax.bar(x + w/2, clf_results['CV Accuracy'], w,
               label='CV Accuracy (5-Fold)', color='#FF9800', edgecolor='black')

ax.set_xticks(x)
ax.set_xticklabels([f"{row['Model']}\n({row['Task']})" for _, row in clf_results.iterrows()], fontsize=9)
ax.set_ylabel('Accuracy')
ax.set_title('Classification Models: Accuracy Comparison', fontsize=15, fontweight='bold')
ax.legend(fontsize=12)
ax.set_ylim(0, 1)

ax.bar_label(bars1, fmt='%.3f', fontsize=9)
ax.bar_label(bars2, fmt='%.3f', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Regression models comparison
reg_data = {
    'Model': ['Linear Regression', 'Random Forest Regressor'],
    'MAE': [mae_lin, mae_rfr],
    'RMSE': [rmse_lin, rmse_rfr],
    'R2': [r2_lin, r2_rfr]
}
reg_df = pd.DataFrame(reg_data)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, metric in enumerate(['MAE', 'RMSE', 'R2']):
    colors = ['#2196F3', '#4CAF50']
    bars = axes[i].bar(reg_df['Model'], reg_df[metric], color=colors, edgecolor='black')
    axes[i].set_title(metric, fontsize=14, fontweight='bold')
    axes[i].bar_label(bars, fmt='%.3f', fontsize=11, fontweight='bold')
    axes[i].tick_params(axis='x', labelsize=9)

fig.suptitle('Regression Models: Performance Comparison', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Predict the Next El Clasico

In [ ]:
# Use the latest data point to predict the next match
latest = ml_ready.iloc[-1:]
X_next = scaler.transform(latest[FEATURES].values)

print('NEXT EL CLASICO PREDICTION')
print('=' * 50)
print(f'Based on features from: {latest["date_str"].values[0]}')
print()

# Each model predicts
pred_lr = le_result.inverse_transform(lr.predict(X_next))[0]
pred_rf = le_result.inverse_transform(rf.predict(X_next))[0]

result_map = {'H': 'Home Win', 'D': 'Draw', 'A': 'Away Win'}

print(f'Logistic Regression: {result_map.get(pred_lr, pred_lr)}')
print(f'Random Forest:       {result_map.get(pred_rf, pred_rf)}')

pred_knn = le_rm.inverse_transform(knn.predict(X_next))[0]
pred_svm = le_rm.inverse_transform(svm.predict(X_next))[0]
print(f'KNN:                 {pred_knn}')
print(f'SVM:                 {pred_svm}')

pred_goals_lin = lin_reg.predict(X_next)[0]
pred_goals_rf = rf_reg.predict(X_next)[0]
print()
print(f'Predicted total goals (Linear Reg):  {pred_goals_lin:.1f}')
print(f'Predicted total goals (RF Reg):      {pred_goals_rf:.1f}')

# Probability from Logistic Regression
proba = lr.predict_proba(X_next)[0]
print()
print('Win probabilities (Logistic Regression):')
for cls, prob in zip(le_result.classes_, proba):
    print(f'  {result_map.get(cls, cls):12s}: {prob*100:.1f}%')

---
## Summary

### Key Findings

1. **Historical form** (rolling win rates) is the most important predictor
2. **Home advantage** is a significant factor in El Clasico
3. **Draws are hardest to predict**: all models struggle with the Draw class
4. **Random Forest** tends to outperform other classifiers
5. **Goal prediction** is inherently noisy: football matches have high variance

### Models Used

| Type | Models |
|------|--------|
| Classification | Logistic Regression, Random Forest, KNN, SVM, Decision Tree |
| Regression | Linear Regression, Random Forest Regressor |

### Techniques Applied

| Area | What We Did |
|------|-------------|
| Feature Engineering | Rolling averages, cumulative stats, encoding |
| End-to-end Pipeline | ML pipeline on scraped data |